# ATRIUM Task 4.1.3 Information Extraction on ATHENA JSONL records

This notebook demonstrates applying the USW vocabulary-based information extraction pipeline to records in a JSONL data file supplied by ATHENA, and merging the output results for further subsequent processing.
The pipeline identifies terms originating from the following three Linked Open Data controlled vocabularies:

* [FISH Event Types Thesaurus](http://purl.org/heritagedata/schemes/agl_et)
* [FISH Archaeological Sciences Thesaurus](http://purl.org/heritagedata/schemes/560)
* [Getty Art & Architecture Thesaurus - Activities facet](https://vocab.getty.edu/aat/300404112)

The [Forum on Information Standards in Heritage (FISH)](https://heritage-standards.org.uk/) terminology working group administer and maintain UK national standard controlled terminologies for cultural heritage - known as the [FISH Vocabularies](https://heritage-standards.org.uk/fish-vocabularies/). These are made freely available for download and use as Linked Open Data via the [Heritage Data](https://www.heritagedata.org/) site. 

The [Getty Art & Architecture Thesaurus (AAT)](http://vocab.getty.edu/aat/) is made available online as Linked Open Data. The (poly)hierarchical structure is subdivided at the top level into a number of facets. We have isolated terms originating from the [AAT Activities facet](https://vocab.getty.edu/aat/300404112) for this exercise.

The listing following the source code sections below shows actual output of the process. These results are also merged within the originating input data under the 'spans' section and saved as a new file. The following example illustrates how the spans identified in the 3rd result record are merged into the existing 'spans' section:

![alt text](img/added-span-results.png "Added span results")



In [3]:
%%capture
import warnings
# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)
warnings.filterwarnings(action='ignore', category=FutureWarning)

# install prerequisites
%pip install spacy
%pip install srsly

%sx python -m spacy download en_core_web_sm

In [4]:
import spacy # for text processing
# from spacy import displacy # for visualisation of tagged text
from spacy.tokens import Span
import srsly # for JSONL serialization/deserialization
import os
import json
from slugify import slugify # for valid filenames
from components import DocSummary # custom vocabulary-based components
from components.Util import read_json_file # for reading supplementary lists from JSON files
from IPython.display import display, HTML


# check if a given span exists in a list of spans
# comparing start/end positions and label
def span_exists(span: dict, lst: list) -> bool:
   return any(
        item["start"] == span.get("start", 0)
        and item["end"] == span.get("end", 0) 
        and item["label"] == span.get("label", "") for item in lst
    )           


if __name__ == '__main__':  

    # default vocabulary patterns defined here
    # these may be overriden by local files 
    vocab_folder = "./vocabularies"

    # set up default base pipeline (English)
    nlp = spacy.load("en_core_web_sm", disable = ['ner'])
    supp_list_act = read_json_file(f"{vocab_folder}/supp_list_AAT_ACTIVITIES.json")
    
    # add custom pipeline components
    nlp.add_pipe("text_normalizer", before = "tagger")

    # object types (from FISH object types vocabulary)
    nlp.add_pipe(
        "vocabulary_ruler", 
        name = "fish_event_types_ruler",
        last = True, 
        config = {
            "default_label": "FISH_EVENT",
            "lemmatize": True,
            "min_lemm_length": 4,
            "min_term_length": 3,
            "patt_list": read_json_file(f"{vocab_folder}/patterns_FISH_agl_et_20260513.json")
        }
    ) 

    nlp.add_pipe(
        "vocabulary_ruler", 
        name = "fish_arch_sciences_ruler",
        last = True, 
        config = {
            "default_label": "FISH_ARCHSCIENCE",
            "lemmatize": True,
            "min_lemm_length": 4,
            "min_term_length": 3,
            "patt_list": read_json_file(f"{vocab_folder}/patterns_FISH_560_20260513.json")
        }
    ) 

    nlp.add_pipe(
        "vocabulary_ruler", 
        name = "fish_sctivities_ruler",
        last = True, 
        config = {
            "default_label": "AAT_ACTIVITY",
            "lemmatize": True,
            "min_lemm_length": 4,
            "min_term_length": 3,
            "patt_list": read_json_file(f"{vocab_folder}/patterns_AAT_ACTIVITIES_20231018.json")
        }
    ) 
    
    nlp.add_pipe("child_span_remover", last=True) 
    
     # read JSONL input data from file
    input_data_path = "./data/athena"
    input_file_name = "sample_annotated_output.jsonl"   
    input_file_path = os.path.join(input_data_path, input_file_name) 
    data: list = list(srsly.read_jsonl(input_file_path))

    # process each item in the input data
    for item in data:
        identifier = item.get("meta", {}).get("id", "").strip()
        text = item.get("text", "")
        # run pipeline against input text
        doc = nlp(text)
        
        # display HTML summary of results (see below)
        summary = DocSummary(doc)
        display(HTML(f"<h3>[ID: {identifier}]</h3>"))
        if(len(summary.spans) == 0):
            display(text)
        else:
            display(HTML(summary.doctext_to_html()))
        
        display(HTML(summary.spans_to_html()))
        #display(HTML(summary.tokens(format="html")))
        display(HTML("<hr>"))

        # add new spans to the existing spans array,
        # checking for duplicates (in case multiple runs)
        the_spans: list = item.get("spans", []) 
        new_spans = summary.spans_to_list()
        for span in new_spans:
            if not span_exists(span, the_spans):
                the_spans.append(span)
        item["spans"] = the_spans
        #item["tokens2"] = summary.tokens_to_list()
    
    # create output file path if it does not already exist
    output_data_path = os.path.join(input_data_path, "output")
    if not os.path.exists(output_data_path):
        os.makedirs(output_data_path)

    # output the modified structure to a (new) JSONL file    
    output_file_path = os.path.join(output_data_path, f"{slugify(input_file_name)}-plus-vocab-ner.jsonl") 
    srsly.write_jsonl(output_file_path, data) 


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
22,37,2,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300223990,spatial analysis,0.0,,0.0,0.0,,,spatial analysis
75,79,10,10,AAT_ACTIVITY,http://vocab.getty.edu/aat/300412133,scale,0.0,,0.0,0.0,,,scale
81,90,11,11,AAT_ACTIVITY,http://vocab.getty.edu/aat/300248891,patterning,0.0,,0.0,0.0,,,patterning
177,182,28,28,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053130,models,0.0,,0.0,0.0,,,models
362,375,58,58,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054328,archaeological,0.0,,0.0,0.0,,,archaeological
377,382,59,59,AAT_ACTIVITY,http://vocab.getty.edu/aat/300077610,record,0.0,,0.0,0.0,,,record


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
0,6,0,0,AAT_ACTIVITY,http://vocab.getty.edu/aat/300379429,Samples,0.0,,0.0,0.0,,,Samples
13,21,2,2,AAT_ACTIVITY,http://vocab.getty.edu/aat/300077121,collected,0.0,,0.0,0.0,,,collected
69,78,9,9,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/142118,carbonised,0.0,,0.0,0.0,,,carbonised
89,92,12,12,AAT_ACTIVITY,http://vocab.getty.edu/aat/300445429,Fall,0.0,,0.0,0.0,,,Fall
101,105,16,16,AAT_ACTIVITY,http://vocab.getty.edu/aat/300162124,Lines,0.0,,0.0,0.0,,,Lines
173,181,41,41,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/171049,processed,0.0,,0.0,0.0,,,processed
195,203,44,44,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/142141,flotation,0.0,,0.0,0.0,,,flotation


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
41,63,7,8,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/142158,magnetic susceptibility,0.0,,0.0,0.0,,,magnetic susceptibility
101,107,14,14,AAT_ACTIVITY,http://vocab.getty.edu/aat/300137584,provide,0.0,,0.0,0.0,,,provide
120,126,16,16,AAT_ACTIVITY,http://vocab.getty.edu/aat/300077610,records,0.0,,0.0,0.0,,,records
131,137,18,18,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054116,erosion,0.0,,0.0,0.0,,,erosion
217,225,32,32,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054394,histories,0.0,,0.0,0.0,,,histories


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
12,15,2,2,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/142109,bone,0.0,,0.0,0.0,,,bone
17,23,3,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300379429,samples,0.0,,0.0,0.0,,,samples
72,74,14,14,AAT_ACTIVITY,http://vocab.getty.edu/aat/300264220,AMS,0.0,,0.0,0.0,,,AMS
76,92,15,16,FISH_ARCHSCIENCE,http://purl.org/heritagedata/schemes/560/concepts/142188,radiocarbon dated,0.0,,0.0,0.0,,,radiocarbon dated
136,141,23,23,AAT_ACTIVITY,http://vocab.getty.edu/aat/300404795,Dating,0.0,,0.0,0.0,,,Dating
162,169,27,27,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054687,Research,0.0,,0.0,0.0,,,Research
218,221,37,37,AAT_ACTIVITY,http://vocab.getty.edu/aat/300404795,date,0.0,,0.0,0.0,,,date


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
0,9,0,0,AAT_ACTIVITY,http://vocab.getty.edu/aat/300380461,Calibrated,0.0,,0.0,0.0,,,Calibrated
11,13,1,1,AAT_ACTIVITY,http://vocab.getty.edu/aat/300264220,AMS,0.0,,0.0,0.0,,,AMS
27,32,3,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300194584,assays,0.0,,0.0,0.0,,,assays
190,216,31,33,AAT_ACTIVITY,http://vocab.getty.edu/aat/300081742,neutron activation analysis,0.0,,0.0,0.0,,,neutron activation analysis


...

start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
16,23,3,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300226216,examined,0.0,,0.0,0.0,,,examined
39,50,6,6,AAT_ACTIVITY,http://vocab.getty.edu/aat/300379416,petrographic,0.0,,0.0,0.0,,,petrographic
52,58,7,7,AAT_ACTIVITY,http://vocab.getty.edu/aat/300343813,studies,0.0,,0.0,0.0,,,studies
69,73,10,10,AAT_ACTIVITY,http://vocab.getty.edu/aat/300404350,stone,0.0,,0.0,0.0,,,stone
111,117,17,17,AAT_ACTIVITY,http://vocab.getty.edu/aat/300343813,studies,0.0,,0.0,0.0,,,studies


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
14,19,2,2,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053130,models,0.0,,0.0,0.0,,,models
47,50,10,10,AAT_ACTIVITY,http://vocab.getty.edu/aat/300379469,soil,0.0,,0.0,0.0,,,soil
115,118,28,28,AAT_ACTIVITY,http://vocab.getty.edu/aat/300222731,five,0.0,,0.0,0.0,,,five
120,131,29,29,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053578,measurements,0.0,,0.0,0.0,,,measurements
164,170,38,38,AAT_ACTIVITY,http://vocab.getty.edu/aat/300379429,samples,0.0,,0.0,0.0,,,samples


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
32,39,3,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054595,analysis,0.0,,0.0,0.0,,,analysis
127,134,17,17,AAT_ACTIVITY,http://vocab.getty.edu/aat/300236367,adjusted,0.0,,0.0,0.0,,,adjusted
145,148,20,20,AAT_ACTIVITY,http://vocab.getty.edu/aat/300247929,type,0.0,,0.0,0.0,,,type
153,160,22,22,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054687,research,0.0,,0.0,0.0,,,research


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
13,34,3,5,AAT_ACTIVITY,http://vocab.getty.edu/aat/300081693,trace element analysis,0.0,,0.0,0.0,,,trace element analysis
67,80,11,11,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054328,archaeological,0.0,,0.0,0.0,,,archaeological
92,94,15,15,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053599,can,0.0,,0.0,0.0,,,can
218,228,38,38,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053892,transported,0.0,,0.0,0.0,,,transported
268,277,44,44,AAT_ACTIVITY,http://vocab.getty.edu/aat/300417277,prehistory,0.0,,0.0,0.0,,,prehistory


start,end,token_start,token_end,label,id,text,sec_score,sections,sig_proximity,score,score_explain,context,span
0,6,0,0,AAT_ACTIVITY,http://vocab.getty.edu/aat/300239496,Running,0.0,,0.0,0.0,,,Running
20,24,3,3,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053130,Model,0.0,,0.0,0.0,,,Model
30,37,5,5,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053130,modelled,0.0,,0.0,0.0,,,modelled
49,56,8,8,AAT_ACTIVITY,http://vocab.getty.edu/aat/300054595,analyzed,0.0,,0.0,0.0,,,analyzed
76,79,13,13,AAT_ACTIVITY,http://vocab.getty.edu/aat/300239496,runs,0.0,,0.0,0.0,,,runs
96,100,17,17,AAT_ACTIVITY,http://vocab.getty.edu/aat/300053130,model,0.0,,0.0,0.0,,,model
